# 3. Measuring: one period per sector, the wander, and the floor

This is the whole method. Four steps, each one function in `helpers.py`:

1. **`load_sector`** — read one sector's brightness curve.
2. **`fit_period`** — find the single steady wave that fits it best, searching only between 0.77 and 1.43 × the catalogue period (so a two-spots-opposite star cannot fool us into half the period). Its period is the reading for that sector. Also returned: the wave's height and the **leftover** (data minus wave).
3. **`angular_mad`** — the **wander**: convert each sector's period to angular speed (2π/P, radians per day) and take the robust scatter across sectors.
4. **The floor**, inside **`measure_star`** — the control. We cannot tell from one star whether its readings scatter because the star changed or because the telescope's noise and gaps jiggled the fit. So for each star we build a *perfectly steady fake star*: a wave at the catalogue period with the star's own height, planted in the star's own timestamps, with the star's own leftover added back as the noise. We measure the fake with the same code; its scatter is what "no change at all" looks like for this star. We do this twelve times and take the median.

The leftover is added back in two ways, and both are kept: **shuffled** in time (destroys every slow wiggle — the "white" floor, a lower limit on the instrument's share) and **slid** in time (keeps the slow wiggles — the "shifted" floor, an upper limit, and the one every headline number uses).

The **excess** is what the real star does beyond its floor: as a ratio (wander ÷ floor) and in quadrature (√(wander² − floor²), in rad/day).

In [1]:
import sys; sys.path.insert(0, "..")
import json, time, os
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import helpers as h
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [2]:
# One star first: the Sun-like example.
stars = {r["tic_id"]: r for r in h.read_star_list("original_450")}
cache = h.DATA / "lightcurves" / "original"
t0 = time.time()
one = h.measure_star(stars[45468124], cache)
print(f"took {time.time()-t0:.1f} s")
periods = {int(k): float(v) for k, v in (item.split(":") for item in one["sector_periods"].split(","))}
print("period read in each sector (days):", {k: round(v, 3) for k, v in periods.items()})
print(f"wander       = {one['wander']:.4f} rad/day  (robust scatter of 2π/P across the {one['sectors']} sectors)")
print(f"white floor  = {one['floor_white']:.4f} rad/day  -> ratio {one['ratio_white']:.1f}")
print(f"shifted floor= {one['floor_shifted']:.4f} rad/day  -> ratio {one['ratio_shifted']:.1f}")

took 5.9 s
period read in each sector (days): {8: 2.777, 9: 2.895, 10: 2.857, 35: 2.637, 36: 2.885, 37: 3.131, 62: 2.953, 63: 2.958, 89: 3.005, 90: 3.012}
wander       = 0.0802 rad/day  (robust scatter of 2π/P across the 10 sectors)
white floor  = 0.0016 rad/day  -> ratio 51.2
shifted floor= 0.0218 rad/day  -> ratio 3.7


In [3]:
# Now every star in both samples. Each star is independent, so we use several CPU cores.
from concurrent.futures import ProcessPoolExecutor

def measure_sample(sample_name, cache_name, out_csv):
    out_csv = Path(out_csv)
    if out_csv.exists():
        print(f"{out_csv.name} already exists - skipping (delete it to recompute)")
        return pd.read_csv(out_csv)
    rows = h.read_star_list(sample_name)
    cache = h.DATA / "lightcurves" / cache_name
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=max(1, (os.cpu_count() or 2) - 2)) as pool:
        results = list(pool.map(h.measure_star, rows, [cache] * len(rows), chunksize=4))
    table = pd.DataFrame([r for r in results if r is not None])
    table.to_csv(out_csv, index=False)
    print(f"{sample_name}: {len(table)} of {len(rows)} stars measured in {(time.time()-t0)/60:.1f} min -> {out_csv.name}")
    return table

original = measure_sample("original_450", "original", "../results/stars_original.csv")
replication = measure_sample("replication_422", "replication", "../results/stars_replication.csv")

stars_original.csv already exists - skipping (delete it to recompute)
stars_replication.csv already exists - skipping (delete it to recompute)


In [4]:
for name, table in [("original", original), ("replication", replication)]:
    print(f"{name}: {len(table)} stars, {int(table.sectors.sum())} sectors; "
          f"median wander {table.wander.median():.4f}, median white floor {table.floor_white.median():.4f}, "
          f"median shifted floor {table.floor_shifted.median():.4f} rad/day")
original[["tic_id", "teff", "period", "sectors", "wander", "floor_white", "floor_shifted", "ratio_shifted", "excess_shifted"]].head(8).round(4)

original: 450 stars, 4317 sectors; median wander 0.0179, median white floor 0.0023, median shifted floor 0.0090 rad/day
replication: 422 stars, 4016 sectors; median wander 0.0157, median white floor 0.0026, median shifted floor 0.0094 rad/day


,tic_id,teff,period,sectors,wander,floor_white,floor_shifted,ratio_shifted,excess_shifted
0,3544322,3018.0,4.1559,9,0.0295,0.0016,0.0087,3.3839,0.0282
1,31869488,3186.5,5.6584,15,0.0096,0.0046,0.0068,1.4010,0.0067
2,142319027,3197.1,1.8254,9,0.0119,0.0072,0.0073,1.6290,0.0094
3,154009241,3151.7,4.3203,9,0.0134,0.0044,0.0067,1.9859,0.0115
4,159652320,3178.8,3.8170,9,0.0119,0.0049,0.0086,1.3905,0.0083
5,160474395,3128.7,4.2196,14,0.0165,0.0095,0.0117,1.4155,0.0117
6,202510001,3145.3,2.7244,10,0.0081,0.0083,0.0098,0.8240,0.0000
7,219085997,3184.7,2.6831,8,0.0378,0.0094,0.0135,2.8062,0.0353


**What to notice.** Each row is one star. `wander` is what the star's readings did; the two floors are what a steady star would have done in its data; `ratio_shifted` above 1 means the star's readings moved more than a steady star's would. Notebook 4 asks how these depend on temperature.